# Day 24 — Lecture 12: ElasticNet Overview

> Source: Apna College Prime AI/ML • Day 24 Lecture 12 • Video Duration: `03:56`
> Core Objective: `ElasticNet` samajhna jo `l1_ratio` parameter se Lasso (`L1`) aur Ridge (`L2`) ko combine karta hai.
> Key Parameters: `alpha`, `l1_ratio` (`0` = Ridge, `1` = Lasso, `0.5` = 50-50 blend)
> Dataset Used: `insurance.csv`
> Core Concept: ElasticNet cost function me `l1_ratio` batata hai ki kitna percent L1 penalty aur `(1 - l1_ratio)` kitna L2 penalty rakhni hai

## 🛠️ Step 0: Setup & Insurance Train-Test Data

Chalo `ElasticNet` import karke `insurance.csv` data prepare karte hain taaki `l1_ratio` ka effect live dekh sakein.

In [ ]:
# 1. Libraries aur ElasticNet import karo
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 2. Data load aur feature engineering apply karo
insurance_data = pd.read_csv("insurance.csv")
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]
X = pd.get_dummies(X, columns=["region"], drop_first=True, dtype=int)
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 📍 Step 1: ElasticNet Cost Equation & `l1_ratio` Logic

ElasticNet dono penalties ko blend karta hai:
- **Cost Penalty:** `alpha * l1_ratio * L1 + 0.5 * alpha * (1 - l1_ratio) * L2`

| `l1_ratio` Value | Active Penalty | Equivalent Model |
|:---:|---|---|
| `0` | Sirf `L2` (`(1 - 0) = 1`) | **Ridge Regression** |
| `0.5` | 50% `L1` + 50% `L2` | **Balanced ElasticNet** |
| `1` | Sirf `L1` (`(1 - 1) = 0`) | **Lasso Regression** |

In [ ]:
# 1. 50-50 mix (l1_ratio=0.5) ke saath ElasticNet fit karo
enet = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000, random_state=42)
enet.fit(X_train, y_train)

# 2. Test predictions aur metrics check karo
y_pred = enet.predict(X_test)
print("ElasticNet (0.5) MSE:", mean_squared_error(y_test, y_pred))
print("ElasticNet (0.5) R2 :", r2_score(y_test, y_pred))

## 🎯 Step 2: Compare Different `l1_ratio` Blends (`0.01`, `0.5`, `1.0`)

Dekho kaise `l1_ratio` ko `0` ke paas se `1` ki taraf le jaane par L2 (Ridge) se L1 (Lasso) ki taraf behavior shift hota hai.

In [ ]:
# 1. Alag-alag l1_ratio values par ElasticNet test karo
for ratio in [0.01, 0.5, 1.0]:
    m = ElasticNet(alpha=1.0, l1_ratio=ratio, max_iter=5000, random_state=42)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    zero_w = int(np.sum(np.abs(m.coef_) < 1e-6))
    print(f"l1_ratio={ratio:<4} -> R2: {r2_score(y_test, pred):.4f} | Zero weights: {zero_w}")

## 🏆 Step 3: Regression Section Wrap-up

Lecture 12 ke saath humne Regression ka poora foundation complete kiya:
- **Models:** Linear Regression, Lasso (`L1`), Ridge (`L2`), ElasticNet (`L1 + L2`).
- **Core Concepts:** Feature Engineering, Dummy Variable Trap, Bias-Variance Trade-off, Cross Validation (`LassoCV`).

In [ ]:
# 1. Balanced ElasticNet model ke learned feature weights dekho
for col, coef in zip(X.columns, enet.coef_):
    print(f"{col:<18}: {coef:10.2f}")